# 11b. Data Leakage Investigation (Cα + hierarchical tree)

The FoldMason guide-tree split used in MI parameter scans may be circular: the Newick tree comes from structural alignment of the same PDBs that produce the Cα distance matrix. If alignment geometry and Cα distances encode related information, group-aware splits may not be independent of the features.

This notebook probes that circularity by replacing the FoldMason Newick with a **Ward hierarchical clustering tree built directly from the Cα feature matrix**, then running the same `run_mi_parameter_scan_with_guide_tree` machinery.

**August adaptations**
- Prefer local `ca_corr_filtered_*` / `ca_filtered_*` / `ca_feature_matrix.csv` (+ matching labels/pickle).
- If a mustang seed directory is missing, shuffle all matrix structures (`seed=42`) and take cumulative prefixes.
- Clamp nominal sizes `[500, 1000, 1500, 2000, 2523]` to the available row count.
- Outputs under `Results/data_leakage/`.


## Table of contents

0. [Setup & Paths](#setup)
1. [Load Cα Feature Matrix](#load)
2. [Hierarchical Clustering → Newick Export](#tree)
3. [MI Parameter Scans](#scans)
4. [Load Pre-computed Scan Results](#load-scans)
5. [Comparison: Performance vs Dataset Size](#comparison)


In [ ]:
import os
import sys
import random
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.cluster.hierarchy import linkage, to_tree
from scipy.spatial.distance import pdist

from workflow.feature_classification import FeatureClassification

print("Imports OK")


---

## 0. Setup & Paths  <a id="setup"></a>


In [ ]:
SEED = 42

def _first_existing(candidates):
    for p in candidates:
        if p and os.path.isfile(p):
            return p
    return None

def _find_mustang_input():
    candidates = [
        "Results/structural_alignment/mustang/input",
        "Results/Experiments/structural_alignment/mustang/input",
    ]
    candidates.extend(sorted(glob("Results/Experiments/structural_alignment/**/mustang/input", recursive=True)))
    for d in candidates:
        if os.path.isdir(d) and any(f.endswith(".pdb") for f in os.listdir(d)):
            return d
    return None

# Resolve Cα matrix / labels / optional reference pickle from August cwd artifacts
CA_FEATURE_CSV = _first_existing([
    "ca_corr_filtered_feature_matrix.csv",
    "ca_filtered_feature_matrix.csv",
    "ca_feature_matrix.csv",
])
if CA_FEATURE_CSV is None:
    raise FileNotFoundError(
        "No Cα feature matrix found. Expected one of: "
        "ca_corr_filtered_feature_matrix.csv, ca_filtered_feature_matrix.csv, ca_feature_matrix.csv"
    )

_prefix = CA_FEATURE_CSV.replace("feature_matrix.csv", "")
CA_LABELS_CSV = _first_existing([
    f"{_prefix}labels.csv",
    "ca_corr_filtered_labels.csv",
    "ca_filtered_labels.csv",
    "ca_labels.csv",
])
if CA_LABELS_CSV is None:
    raise FileNotFoundError(f"No labels CSV found for matrix {CA_FEATURE_CSV}")

CA_REFERENCE_PKL = _first_existing([
    f"{_prefix}reference_data.pkl",
    "ca_corr_filtered_reference_data.pkl",
    "ca_filtered_reference_data.pkl",
    "ca_reference_data.pkl",
])

MUSTANG_INPUT_DIR = _find_mustang_input()
RECONSTR_DIR = "Results/activation_segments/reconstr_MODELLER_aligned"
if not os.path.isdir(RECONSTR_DIR):
    RECONSTR_DIR = None

EXPERIMENT_DIR = "Results/data_leakage"
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

DATASET_SIZES = [500, 1000, 1500, 2000, 2523]

print(f"Cα matrix     : {CA_FEATURE_CSV}")
print(f"Labels        : {CA_LABELS_CSV}")
print(f"Reference pkl : {CA_REFERENCE_PKL or '(none — pairs parsed from columns)'}")
print(f"Mustang input : {MUSTANG_INPUT_DIR or '(none — shuffle-all seed)'}")
print(f"Reconstr dir  : {RECONSTR_DIR or '(optional, missing)'}")
print(f"Output dir    : {EXPERIMENT_DIR}")


---

## 1. Load Cα Feature Matrix  <a id="load"></a>

Load the Cα feature matrix and labels. Build an ordered structure list: mustang/input matches as the seed set when available; otherwise shuffle all rows (`seed=42`). Larger sizes append the remaining structures in that fixed order.


In [ ]:
feature_df = pd.read_csv(CA_FEATURE_CSV, index_col=0)
labels_df = pd.read_csv(CA_LABELS_CSV)

print(f"Full Cα feature matrix : {feature_df.shape}  (structures × features)")
_lab = labels_df["label"].values if "label" in labels_df.columns else labels_df.iloc[:, -1].values
print(f"Label distribution     : {dict(zip(*np.unique(_lab, return_counts=True)))}")


In [ ]:
all_structures = list(feature_df.index)

if MUSTANG_INPUT_DIR:
    mustang_names = set(
        os.path.splitext(f)[0]
        for f in os.listdir(MUSTANG_INPUT_DIR)
        if f.endswith(".pdb")
    )
    # also allow names that include _aligned suffixes by exact stem match after normalize-ish
    seed = sorted([s for s in all_structures if s in mustang_names or os.path.splitext(str(s))[0] in mustang_names])
    rest = sorted([s for s in all_structures if s not in seed])
    random.seed(SEED)
    random.shuffle(rest)
    ordered_names = seed + rest
    print(f"Seed set (mustang/input matches) : {len(seed)}")
    print(f"Remaining structures             : {len(rest)}")
else:
    ordered_names = list(all_structures)
    random.seed(SEED)
    random.shuffle(ordered_names)
    print("No mustang/input found — using shuffle-all ordering (seed=42)")

print(f"Total in feature matrix          : {len(all_structures)}")

DATASET_SIZES = [n for n in DATASET_SIZES if n <= len(ordered_names)]
if len(ordered_names) not in DATASET_SIZES:
    DATASET_SIZES.append(len(ordered_names))
DATASET_SIZES = sorted(set(DATASET_SIZES))

print(f"\nDataset sizes to scan : {DATASET_SIZES}")


---

## 2. Hierarchical Clustering → Newick Export  <a id="tree"></a>

For each dataset size:
1. Subsample rows from the full Cα matrix.
2. Impute NaNs with column medians.
3. Build a **Ward** dendrogram on Euclidean distances of z-scored features.
4. Export Newick with leaf labels matching structure names (drop-in for FoldMason `msa.nw`).


### 2.1 Newick helper  <a id="tree-helper"></a>


In [ ]:
def linkage_to_newick(Z: np.ndarray, labels: list) -> str:
    """Convert a scipy linkage matrix Z to a Newick-format string."""
    tree, _ = to_tree(Z, rd=True)

    def _recurse(node) -> str:
        if node.is_leaf():
            return str(labels[node.id])
        left = _recurse(node.get_left())
        right = _recurse(node.get_right())
        dist = node.dist / 2.0
        return f"({left}:{dist:.6f},{right}:{dist:.6f})"

    return _recurse(tree) + ";"


def build_hclust_newick(
    sub_df: pd.DataFrame,
    method: str = "ward",
    metric: str = "euclidean",
) -> str:
    """Impute NaNs, z-score features, Ward-link, return Newick with row-index leaves."""
    X = sub_df.values.copy().astype(float)
    col_medians = np.nanmedian(X, axis=0)
    nan_mask = np.isnan(X)
    X[nan_mask] = np.take(col_medians, np.where(nan_mask)[1])
    col_std = X.std(axis=0)
    col_std[col_std == 0] = 1.0
    X = (X - X.mean(axis=0)) / col_std
    condensed = pdist(X, metric=metric)
    Z = linkage(condensed, method=method)
    return linkage_to_newick(Z, list(sub_df.index))


print("Newick helper functions defined.")


### 2.2 Build trees for all dataset sizes  <a id="tree-build"></a>

Write `Results/data_leakage/n{size}/hclust.nw` plus matching `ca_feature_matrix.csv` / `ca_labels.csv` for each size.


In [ ]:
newick_paths = {}
subset_dirs = {}

label_col = "label" if "label" in labels_df.columns else labels_df.columns[-1]
struct_col = "structure" if "structure" in labels_df.columns else labels_df.columns[0]
label_map = dict(zip(labels_df[struct_col], labels_df[label_col]))

for n in DATASET_SIZES:
    subset_names = ordered_names[:n]
    sub_dir = os.path.join(EXPERIMENT_DIR, f"n{n}")
    os.makedirs(sub_dir, exist_ok=True)

    sub_feature_df = feature_df.loc[feature_df.index.isin(subset_names)].copy()
    sub_feature_df = sub_feature_df.reindex([s for s in subset_names if s in sub_feature_df.index])

    feat_csv = os.path.join(sub_dir, "ca_feature_matrix.csv")
    sub_feature_df.to_csv(feat_csv)

    sub_labels = pd.DataFrame({
        struct_col: list(sub_feature_df.index),
        label_col: [label_map.get(s, -1) for s in sub_feature_df.index],
    })
    lbl_csv = os.path.join(sub_dir, "ca_labels.csv")
    sub_labels.to_csv(lbl_csv, index=False)

    print(f"n={n:4d} — building Ward dendrogram on {sub_feature_df.shape} ... ", end="", flush=True)
    newick_str = build_hclust_newick(sub_feature_df)
    nw_path = os.path.join(sub_dir, "hclust.nw")
    with open(nw_path, "w") as fh:
        fh.write(newick_str)
    print(f"done  →  {nw_path}")

    newick_paths[n] = nw_path
    subset_dirs[n] = sub_dir

print("\n✅ All subset CSVs and Newick files written.")


---

## 3. MI Parameter Scans  <a id="scans"></a>

Each scan uses the hierarchical-clustering Newick for group-aware train/test splits and the subset CSVs from §2. Reference pickle is optional (pairs are parsed from `i-j` columns when missing).

Scans can be long; §4 reloads checkpoints if already computed.


In [ ]:
scans = {}

# Cap n_features ladder to matrix width
_n_feat_max = feature_df.shape[1]
_n_features_values = [k for k in range(300, 2001, 200) if k <= _n_feat_max]
if _n_feat_max not in _n_features_values and _n_feat_max >= 50:
    _n_features_values.append(_n_feat_max)
_n_features_values = sorted(set(_n_features_values))
print(f"n_features ladder: {_n_features_values}")

for n in DATASET_SIZES:
    print("\n" + "=" * 60)
    print(f"MI scan for n={n}")
    print("=" * 60)
    scans[n] = FeatureClassification.run_mi_parameter_scan_with_guide_tree(
        reference_pickle=CA_REFERENCE_PKL,
        feature_matrix_csv=os.path.join(subset_dirs[n], "ca_feature_matrix.csv"),
        labels_csv=os.path.join(subset_dirs[n], "ca_labels.csv"),
        newick_path=newick_paths[n],
        height_step=2,
        height_min=2,
        height_max=0,
        n_features_values=_n_features_values,
        n_repeats=3,
        train_size=0.9,
        n_estimators=100,
        perm_repeats=100,
        show_plots=True,
        output_dir=os.path.join(subset_dirs[n], "mi_scan"),
    )
    print(f"n={n} best params: height={scans[n]['best_h']}, N={scans[n]['best_n']}")

print("\n✅ All MI scans complete")


---

## 4. Load Pre-computed Scan Results  <a id="load-scans"></a>

Reload checkpoints without re-running §3.


In [ ]:
loaded_scans = {}

for n in DATASET_SIZES:
    sub_dir = os.path.join(EXPERIMENT_DIR, f"n{n}")
    ckpt_dir = os.path.join(sub_dir, "mi_scan", "checkpoints")
    best_csv = os.path.join(sub_dir, "mi_scan", "best_combinations.csv")

    if not os.path.isdir(ckpt_dir) or not os.path.isfile(best_csv):
        print(f"n={n:4d} — no checkpoint / best_combinations found, skipping.")
        continue

    loaded = FeatureClassification.load_mi_scan_results_for_distributions(
        checkpoint_dir=ckpt_dir,
        best_combinations_csv=best_csv,
        reference_pickle=CA_REFERENCE_PKL,
        feature_matrix_csv=os.path.join(sub_dir, "ca_feature_matrix.csv"),
        labels_csv=os.path.join(sub_dir, "ca_labels.csv"),
    )
    loaded_scans[n] = loaded
    print(f"n={n:4d} — loaded. best_h={loaded['best_h']}, best_n={loaded['best_n']}")

# Prefer in-memory scans from §3 when present
try:
    for n, scan in scans.items():
        loaded_scans[n] = scan
except NameError:
    pass

if loaded_scans:
    n_full = max(loaded_scans.keys())
    best_h_dl = loaded_scans[n_full]["best_h"]
    best_n_dl = loaded_scans[n_full]["best_n"]
    X_df_dl = loaded_scans[n_full]["X_df"]
    y_dl = loaded_scans[n_full]["y"]
    Xk_dl = loaded_scans[n_full]["Xk"]
    feature_labels_dl = loaded_scans[n_full]["feature_labels_all"]
    gini_dl = loaded_scans[n_full]["gini_mean"]
    perm_dl = loaded_scans[n_full]["perm_mean"]
    res_df_dl = loaded_scans[n_full].get("res_df")
    print(f"\n✅ Primary variables set from n={n_full}")
else:
    raise RuntimeError("No scan results loaded. Run §3 first.")


---

## 5. Comparison: Performance vs Dataset Size  <a id="comparison"></a>

Collect best validation accuracy / precision per size. Flat or improving curves suggest the hierarchical-tree split is not uniquely inflating performance relative to dataset size.


### 5.1 Accuracy & Precision vs n  <a id="cmp-perf"></a>


In [ ]:
rows = []
for n, scan in loaded_scans.items():
    res = scan.get("res_df")
    if res is None:
        metrics_csv = os.path.join(EXPERIMENT_DIR, f"n{n}", "mi_scan", "metrics_by_height_n.csv")
        if os.path.isfile(metrics_csv):
            res = pd.read_csv(metrics_csv)
        else:
            print(f"n={n}: missing metrics_by_height_n.csv — skip")
            continue

    best = res[(res["height"] == scan["best_h"]) & (res["n_features"] == scan["best_n"])]
    if best.empty:
        best = res.sort_values("accuracy_mean", ascending=False).iloc[[0]]
    rows.append({
        "n": n,
        "best_h": scan["best_h"],
        "best_n": scan["best_n"],
        "accuracy_mean": float(best["accuracy_mean"].values[0]),
        "accuracy_std": float(best["accuracy_std"].values[0]),
        "precision_mean": float(best["precision_mean"].values[0]),
        "precision_std": float(best["precision_std"].values[0]),
    })

perf_df = pd.DataFrame(rows).sort_values("n").reset_index(drop=True)
print(perf_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (ycol, ecol, title) in zip(axes, [
    ("accuracy_mean", "accuracy_std", "Validation Accuracy"),
    ("precision_mean", "precision_std", "Validation Precision"),
]):
    ax.errorbar(
        perf_df["n"], perf_df[ycol], yerr=perf_df[ecol],
        marker="o", capsize=4, color="steelblue",
    )
    ax.set_xlabel("Dataset size (n)")
    ax.set_ylabel(title)
    ax.set_title(f"{title} vs dataset size\n(hierarchical clustering tree)")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, "perf_vs_n.png"), dpi=150)
plt.show()


### 5.2 Feature Stability Across Sizes  <a id="cmp-features"></a>

Jaccard overlap of top-k Gini features across dataset sizes.


In [ ]:
TOP_K = 20

top_features = {}
for n, scan in loaded_scans.items():
    gini = scan["gini_mean"]
    labels = scan["feature_labels_all"]
    top_idx = np.argsort(gini)[::-1][:TOP_K]
    top_features[n] = set(np.array(labels)[top_idx])

sizes = sorted(top_features.keys())
jaccard = np.zeros((len(sizes), len(sizes)))
for i, ni in enumerate(sizes):
    for j, nj in enumerate(sizes):
        inter = len(top_features[ni] & top_features[nj])
        union = len(top_features[ni] | top_features[nj])
        jaccard[i, j] = inter / union if union > 0 else 0.0

jaccard_df = pd.DataFrame(jaccard, index=sizes, columns=sizes)
print(f"Pairwise Jaccard similarity of top-{TOP_K} features:")
print(jaccard_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    jaccard_df, annot=True, fmt=".2f", cmap="Blues",
    vmin=0, vmax=1, ax=ax,
    xticklabels=[f"n={s}" for s in sizes],
    yticklabels=[f"n={s}" for s in sizes],
)
ax.set_title(f"Jaccard similarity — top {TOP_K} features by Gini importance")
plt.tight_layout()
plt.savefig(os.path.join(EXPERIMENT_DIR, "feature_stability_jaccard.png"), dpi=150)
plt.show()


### 5.3 Full-n distributions + MI W/KL  <a id="cmp-dist"></a>

Split labels come from the hierarchical Newick at `best_h` (group-aware). Violins and Wasserstein/KL use August distribution helpers + MI ranking.


In [ ]:
# KinCore bio labels CSV for distribution helpers (optional PCA file)
PCA_CLUSTER_LABELS = "cluster_labels_my_analysis_hierarchical.txt"
if not os.path.isfile(PCA_CLUSTER_LABELS):
    alt = "Results/activation_segments/cluster_labels_my_analysis_hierarchical.txt"
    PCA_CLUSTER_LABELS = alt if os.path.isfile(alt) else PCA_CLUSTER_LABELS

KINCORE_CSV = "Results/dunbrack_assignments/kinase_conformation_assignments.csv"
bio_csv = os.path.join(EXPERIMENT_DIR, "kincore_bio_labels.csv")
if os.path.isfile(KINCORE_CSV):
    from workflow.pca_analysis import ClusterAnalyzer
    ca = ClusterAnalyzer(n_clusters=2)
    bio_labels, _ = ca.load_kincore_labels(list(X_df_dl.index), kincore_file=KINCORE_CSV)
    pd.DataFrame({"structure": list(X_df_dl.index), "label": np.asarray(bio_labels).astype(int)}).to_csv(
        bio_csv, index=False
    )
else:
    # fall back to classification labels as a stand-in so the helper can load a CSV
    pd.DataFrame({"structure": list(X_df_dl.index), "label": np.asarray(y_dl).astype(int)}).to_csv(
        bio_csv, index=False
    )
    print(f"⚠️  KinCore CSV missing; wrote stand-in bio labels from y to {bio_csv}")

split_labels_dl, train_idx_dl, test_idx_dl, _groups = (
    FeatureClassification.split_labels_from_newick_guide_tree(
        structure_names=list(X_df_dl.index),
        newick_path=newick_paths[n_full],
        height=best_h_dl,
        train_size=0.9,
        random_state=42,
        labels=y_dl,
        feature_matrix=Xk_dl,
    )
)

if not os.path.isfile(PCA_CLUSTER_LABELS):
    print(f"⚠️  PCA cluster file missing ({PCA_CLUSTER_LABELS}); skipping cluster violins / WKL")
    distribution_plot_results_dl = None
else:
    distribution_plot_results_dl = FeatureClassification.plot_top_feature_distributions_by_label_and_cluster(
        X_df=X_df_dl,
        Xk=Xk_dl,
        feature_labels_all=feature_labels_dl,
        gini_mean=gini_dl,
        perm_mean=perm_dl,
        best_h=best_h_dl,
        best_k=best_n_dl,
        biological_labels_csv=bio_csv,
        pca_cluster_labels_file=PCA_CLUSTER_LABELS,
        split_labels=split_labels_dl,
        train_idx=train_idx_dl,
        test_idx=test_idx_dl,
        n_top=20,
        title_suffix=f"(hierarchical tree; h={best_h_dl}, n={best_n_dl})",
    )
    print("✅ Feature distribution plots (full-dataset, hierarchical tree) complete")

    N_POOL = int(best_n_dl)
    dl_wkl = FeatureClassification.compute_cluster0_vs_cluster1_wasserstein_kl_for_mi_features(
        X_df=X_df_dl,
        y=y_dl,
        distribution_plot_results=distribution_plot_results_dl,
        n_mi_features=N_POOL,
        n_bins_kl=31,
    )
    wkl_csv = os.path.join(EXPERIMENT_DIR, "wkl_mi_full.csv")
    dl_wkl.to_csv(wkl_csv, index=False)
    FeatureClassification.plot_cluster0_vs_cluster1_wasserstein_kl(
        dl_wkl,
        best_h=best_h_dl,
        best_k=best_n_dl,
        n_pool_features=N_POOL,
        rank_col="mi_rank",
        rank_xlabel="MI feature rank on Cα distances (1 = highest MI)",
        pool_caption="MI-ranked Cα (hierarchical tree split)",
        split_caption="group-aware Newick split",
        save_path=os.path.join(EXPERIMENT_DIR, "wkl_mi_full.png"),
    )
    print(f"✅ Wasserstein / KL complete → {wkl_csv}")

print("=" * 60)
print("✅ 11b DATA LEAKAGE SECTION COMPLETE")
print("=" * 60)
